This is the main YOLO training notebook: train the reversed-order
learning-rate sweep, train the one "optimized" run with the settings I
ended up liking best, run a quick k-fold check to see how stable the
results are, and finally validate everything with the resource monitor.
Each cell below is a separate experiment, they don't depend on each other
running in order except that the k-fold and validation cells expect a
model to already be trained.

Trains YOLOv8n once per learning rate in the list, on the reversed-order
dataset split, keeping every other setting (optimizer, momentum, weight
decay, patience) fixed so the learning rate is the only thing that changes
between runs.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

learning_rate = [0.1, 0.01, 0.001, 0.0001, 0.00001]

for lrnr in learning_rate:
    model.train(
        data="dataset_paper_new/data_nano_rev.yaml",
        epochs=100,
        batch=16,
        imgsz=640,
        device="cuda",
        optimizer="SGD",
        lr0=lrnr,
        momentum=0.937, 
        weight_decay=0.0005,
        patience=5, 
        project="runs_2/train",
        name=f"YOLO_noD_Reversed_lr.{lrnr}",
        exist_ok=True
    )


This is the single "best settings" run - cosine LR schedule, warmup,
higher patience - once I'd already seen which learning rate did well in
the sweep above.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="dataset_paper_new/data_nano.yaml",
    epochs=100,
    batch=16,
    imgsz=640,
    device="cuda",
    workers=4,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.1, 
    momentum=0.937, 
    weight_decay=0.0005,
    cos_lr=True,
    patience=10, 
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    project="runs_2/train",
    name="Optimized_Fall_Model",
    exist_ok=True
)


A 5-fold cross-validation check on the trained model, just to see how much
the metrics move around depending on which slice of the training images end
up as validation. Helps tell a real improvement apart from a lucky split.

In [ ]:
import os
import shutil
import yaml
from sklearn.model_selection import KFold
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

k = 5
DATASET_BASE = "dataset_paper_new"
ORIG_VAL_IMG = f"{DATASET_BASE}/images/train"
ORIG_VAL_LBL = f"{DATASET_BASE}/labels/train"
TEMP_DIR = "temp_kfold_yolo"
MODEL_PATH = "runs_2/train/YOLO_noD_Reversed_lr.0.0001/weights/best.pt"

Path(TEMP_DIR).mkdir(exist_ok=True)

all_files = sorted([f for f in os.listdir(ORIG_VAL_IMG) if f.endswith((".jpg", ".png"))])
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

model = YOLO(MODEL_PATH)

metrics_list = []

for fold, (_, val_idx) in enumerate(kfold.split(all_files), 1):
    print(f"\n🔁 Fold {fold}")

    fold_base = Path(TEMP_DIR) / f"fold{fold}"
    img_dir = fold_base / "images"
    lbl_dir = fold_base / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for i in val_idx:
        fname = all_files[i]
        stem = os.path.splitext(fname)[0]
        shutil.copy(os.path.join(ORIG_VAL_IMG, fname), img_dir / fname)
        shutil.copy(os.path.join(ORIG_VAL_LBL, f"{stem}.txt"), lbl_dir / f"{stem}.txt")

    temp_yaml_path = fold_base / "data_nano_rev.yaml"
    with open(f"{DATASET_BASE}/data_nano_rev.yaml", 'r') as f:
        data_yaml = yaml.safe_load(f)

    data_yaml["val"] = os.path.abspath(img_dir)

    with open(temp_yaml_path, 'w') as f:
        yaml.dump(data_yaml, f)

    results = model.val(
        data=str(temp_yaml_path),
        imgsz=640,
        batch=8,
        iou=0.5,
        conf=0.001,
        verbose=False
    )

    precision = results.box.mp if hasattr(results, "box") else 0.0
    recall = results.box.mr if hasattr(results, "box") else 0.0

    # Compute F1 Score
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0.0

    # Compute Accuracy (approximate)
    if precision + recall + abs(precision - recall) > 0:
        accuracy = (2 * precision * recall) / (precision + recall + abs(precision - recall))
    else:
        accuracy = 0.0

    # Collect metrics
    metrics_list.append({
        "precision": precision,
        "recall": recall,
        "F1 Score": f1,
        "Accuracy": accuracy,
        "mAP50": results.box.map50 if hasattr(results, "box") else 0.0,
        "mAP50-95": results.box.map if hasattr(results, "box") else 0.0
    })
# Cleanup
shutil.rmtree(TEMP_DIR)

# Create DataFrame
df = pd.DataFrame(metrics_list, index=[f"Fold {i}" for i in range(1, k + 1)])
summary = df.agg(['mean', 'std']).rename(index={'mean': 'Mean', 'std': 'Std'})

print("\n=== Per-Fold Metrics ===")
print(df.to_string(float_format="%.4f"))

print("\n=== Cross-Validation Summary ===")
print(summary.to_string(float_format="%.4f"))


Same validation + resource monitoring script as `scripts/yolo_val_only_monitor.py`,
pasted in here so I could run it directly against this notebook's kernel and
GPU without leaving the notebook.

In [ ]:
#!/usr/bin/env python3
"""
This script does NOT train anything. It just loads weights I already trained
for each learning rate (from the runs_2/train/YOLO_noD_Reversed_lr.<lr> folders)
and runs YOLO's validation on them, while a set of callbacks quietly records
how heavy each run was: per-batch inference time, total validation time, CPU
and RAM usage, and GPU utilization / VRAM if NVML is available on the machine.

For every learning rate I get a resource_summary.json and metrics.csv inside
that run's val_monitor folder, and at the end everything gets collected into
one val_monitor_summary.csv so I can compare learning rates side by side.

Needs: ultralytics, psutil, pynvml, pandas, plus torch/torchvision matching
whatever CUDA version is installed.
"""

import os
import json
import time
import statistics as stats
from datetime import datetime
from pathlib import Path

import psutil
import torch
import pandas as pd

# Optional GPU telemetry via NVML
try:
    import pynvml
    _NVML_READY = True
    pynvml.nvmlInit()
except Exception:
    _NVML_READY = False

# Below is the callback that hooks into YOLO's validation loop and collects
# the resource numbers. Ultralytics fires these callbacks itself, I just
# register them on the model.

_PROC = psutil.Process(os.getpid())

def _prime_cpu_percent_samplers():
    try:
        _PROC.cpu_percent(None)
    except Exception:
        pass
    try:
        psutil.cpu_percent(None)
    except Exception:
        pass

def get_cpu_ram_snapshot():
    """Return (proc_cpu%, proc_rss_MB, sys_cpu%, sys_mem%)."""
    try:
        p_cpu = _PROC.cpu_percent(None)
        rss_mb = _PROC.memory_info().rss / (1024**2)
    except Exception:
        p_cpu, rss_mb = None, None
    try:
        sys_cpu = psutil.cpu_percent(None)
        sys_mem = psutil.virtual_memory().percent
    except Exception:
        sys_cpu, sys_mem = None, None
    return p_cpu, rss_mb, sys_cpu, sys_mem

def update_gpu_maxima(max_gpu_util, max_gpu_mem_used):
    """Keep track of the highest GPU util% and VRAM usage seen so far, per GPU.

    Called repeatedly during validation, so these dicts just get overwritten
    with a bigger number whenever we see one.
    """
    if not torch.cuda.is_available():
        return
    try:
        n = torch.cuda.device_count()
    except Exception:
        n = 0
    if n == 0:
        return

    if _NVML_READY:
        for i in range(n):
            try:
                h = pynvml.nvmlDeviceGetHandleByIndex(i)
                util = pynvml.nvmlDeviceGetUtilizationRates(h)
                mem = pynvml.nvmlDeviceGetMemoryInfo(h)
                max_gpu_util[i] = max(max_gpu_util.get(i, 0), int(util.gpu))
                used_mb = mem.used / (1024**2)
                max_gpu_mem_used[i] = max(max_gpu_mem_used.get(i, 0.0), used_mb)
            except Exception:
                pass
    else:
        for i in range(n):
            try:
                used_mb = torch.cuda.memory_allocated(i) / (1024**2)
                max_gpu_mem_used[i] = max(max_gpu_mem_used.get(i, 0.0), used_mb)
            except Exception:
                pass

class ValResourceCallback:
    def __init__(self):
        self.batch_times = []
        self.max_proc_ram_mb = 0.0
        self.max_proc_cpu = 0.0
        self.max_sys_cpu = 0.0
        self.max_sys_mem = 0.0
        self.max_gpu_util = {}      # {gpu_index: percent}
        self.max_gpu_mem_used = {}  # {gpu_index: MB}
        self._t0 = None
        self._val_start = None
        self.summary = {}

    def on_val_start(self, trainer):
        _prime_cpu_percent_samplers()
        if torch.cuda.is_available():
            try:
                torch.cuda.reset_peak_memory_stats()
            except Exception:
                pass
        self._val_start = time.perf_counter()
        update_gpu_maxima(self.max_gpu_util, self.max_gpu_mem_used)

    def on_val_batch_start(self, trainer):
        if torch.cuda.is_available():
            try:
                torch.cuda.synchronize()
            except Exception:
                pass
        self._t0 = time.perf_counter()

    def on_val_batch_end(self, trainer):
        if torch.cuda.is_available():
            try:
                torch.cuda.synchronize()
            except Exception:
                pass
        if self._t0 is not None:
            dt = time.perf_counter() - self._t0
            self.batch_times.append(dt)

        p_cpu, rss_mb, sys_cpu, sys_mem = get_cpu_ram_snapshot()
        if rss_mb is not None:
            self.max_proc_ram_mb = max(self.max_proc_ram_mb, rss_mb)
        if p_cpu is not None:
            self.max_proc_cpu = max(self.max_proc_cpu, p_cpu)
        if sys_cpu is not None:
            self.max_sys_cpu = max(self.max_sys_cpu, sys_cpu)
        if sys_mem is not None:
            self.max_sys_mem = max(self.max_sys_mem, sys_mem)

        update_gpu_maxima(self.max_gpu_util, self.max_gpu_mem_used)

    def on_val_end(self, trainer):
        total_val_time = time.perf_counter() - (self._val_start or time.perf_counter())
        if torch.cuda.is_available():
            try:
                peak_alloc_mb = torch.cuda.max_memory_allocated() / (1024**2)
                peak_res_mb = torch.cuda.max_memory_reserved() / (1024**2)
            except Exception:
                peak_alloc_mb, peak_res_mb = None, None
        else:
            peak_alloc_mb = peak_res_mb = None

        if self.batch_times:
            bt_ms = [t * 1000.0 for t in self.batch_times]
            try:
                p50 = stats.median(bt_ms)
            except Exception:
                p50 = None
            try:
                p95 = stats.quantiles(bt_ms, n=20)[-1] if len(bt_ms) >= 2 else None
            except Exception:
                p95 = None
            avg = sum(bt_ms) / len(bt_ms)
        else:
            avg = p50 = p95 = None

        self.summary = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "num_batches": len(self.batch_times),
            "batch_time_ms_avg": avg,
            "batch_time_ms_p50": p50,
            "batch_time_ms_p95": p95,
            "total_val_time_s": total_val_time,
            "max_process_cpu_percent": self.max_proc_cpu,
            "max_process_ram_mb": self.max_proc_ram_mb,
            "max_system_cpu_percent": self.max_sys_cpu,
            "max_system_mem_percent": self.max_sys_mem,
            "max_gpu_util_percent": self.max_gpu_util,
            "max_gpu_mem_used_mb": self.max_gpu_mem_used,
            "torch_peak_cuda_alloc_mb": peak_alloc_mb,
            "torch_peak_cuda_reserved_mb": peak_res_mb,
            "nvml_available": _NVML_READY,
        }

def run_validation_with_metrics(weights_path, data_yaml, imgsz=640, batch=16, device="cuda", half=True, workers=4, out_dir="val_monitor"):
    """Run model.val() on one set of weights and write the resource + metrics files."""
    from ultralytics import YOLO

    cb = ValResourceCallback()

    model = YOLO(weights_path)

    # Registering callbacks this way instead of passing callbacks= to model.val()
    # because some Ultralytics versions don't accept that kwarg.
    for event, fn in [
        ("on_val_start", cb.on_val_start),
        ("on_val_batch_start", cb.on_val_batch_start),
        ("on_val_batch_end", cb.on_val_batch_end),
        ("on_val_end", cb.on_val_end),
    ]:
        try:
            model.add_callback(event, fn)
        except Exception:
            pass

    results = model.val(
        data=data_yaml,
        imgsz=imgsz,
        batch=batch,
        device=device,
        half=half,
        workers=workers,
        verbose=True,
    )

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Resource summary
    with open(out_dir / "resource_summary.json", "w", encoding="utf-8") as f:
        json.dump(cb.summary, f, indent=2)

    # Ultralytics metrics
    try:
        pd.DataFrame([results.results_dict]).to_csv(out_dir / "metrics.csv", index=False)
    except Exception as e:
        with open(out_dir / "metrics.txt", "w", encoding="utf-8") as f:
            f.write(str(results))

    return cb.summary, out_dir

# Everything below is the actual validation-only run, no training happens here.

def main():
    # Change these if the run layout or dataset config moves.
    DATA_YAML = "dataset_paper_new/data_nano_rev.yaml"
    VAL_BATCH = 16
    IMGSZ = 640
    DEVICE = "cuda"      # or 0 / "cpu"
    PROJECT = "runs_2/train"
    NAME_PREFIX = "YOLO_noD_Reversed_lr"
    USE_HALF_FOR_VAL = True
    # These are the learning rates I already have trained weights for.
    LEARNING_RATES = [0.1, 0.01, 0.001, 0.0001, 0.00001]
    # Prefer best.pt, but last.pt works too if best wasn't saved for some reason.
    PREFER_WEIGHTS = "best.pt"

    project_dir = Path(PROJECT)
    project_dir.mkdir(parents=True, exist_ok=True)

    aggregate_rows = []

    for lr in LEARNING_RATES:
        run_name = f"{NAME_PREFIX}.{lr}"
        run_dir = project_dir / run_name
        weights_dir = run_dir / "weights"

        preferred = weights_dir / PREFER_WEIGHTS
        fallback  = weights_dir / "last.pt"
        if preferred.exists():
            weights_path = preferred
        elif fallback.exists():
            weights_path = fallback
        else:
            print(f"[SKIP] No weights found for lr={lr} at {weights_dir}")
            continue

        print(f"\n=== Validating {weights_path} (lr={lr}) ===\n")
        val_out_dir = run_dir / "val_monitor"

        summary, out_dir = run_validation_with_metrics(
            weights_path=str(weights_path),
            data_yaml=DATA_YAML,
            imgsz=IMGSZ,
            batch=VAL_BATCH,
            device=DEVICE,
            half=USE_HALF_FOR_VAL,
            workers=4,
            out_dir=str(val_out_dir),
        )

        # Load Ultralytics metrics (if available) to aggregate
        metrics_csv = out_dir / "metrics.csv"
        metrics = {}
        if metrics_csv.exists():
            try:
                df = pd.read_csv(metrics_csv)
                metrics = df.iloc[0].to_dict()
            except Exception:
                pass

        # Flatten GPU dicts for convenience (max across devices)
        def _max_or_none(d):
            try:
                return max(d.values()) if isinstance(d, dict) and len(d) > 0 else None
            except Exception:
                return None

        row = {
            "lr0": lr,
            "weights": str(weights_path),
            "run_dir": str(run_dir),
            "total_val_time_s": summary.get("total_val_time_s"),
            "batch_time_ms_avg": summary.get("batch_time_ms_avg"),
            "batch_time_ms_p50": summary.get("batch_time_ms_p50"),
            "batch_time_ms_p95": summary.get("batch_time_ms_p95"),
            "max_process_cpu_percent": summary.get("max_process_cpu_percent"),
            "max_process_ram_mb": summary.get("max_process_ram_mb"),
            "max_system_cpu_percent": summary.get("max_system_cpu_percent"),
            "max_system_mem_percent": summary.get("max_system_mem_percent"),
            "max_gpu_util_percent": _max_or_none(summary.get("max_gpu_util_percent")),
            "max_gpu_mem_used_mb": _max_or_none(summary.get("max_gpu_mem_used_mb")),
            "torch_peak_cuda_alloc_mb": summary.get("torch_peak_cuda_alloc_mb"),
            "torch_peak_cuda_reserved_mb": summary.get("torch_peak_cuda_reserved_mb"),
        }

        # Merge Ultralytics val metrics if present (e.g., precision/recall/mAP50/mAP50-95)
        row.update(metrics)
        aggregate_rows.append(row)

    # Save aggregate CSV at project root for convenience
    agg_path = project_dir / "val_monitor_summary.csv"
    if aggregate_rows:
        pd.DataFrame(aggregate_rows).to_csv(agg_path, index=False)
        print(f"\nSaved aggregate summary: {agg_path.resolve()}")
    else:
        print("\nNo validations run (no weights found). Check your run directories and learning-rate list.")

if __name__ == "__main__":
    main()
